In [64]:
from dotenv import load_dotenv
from typing_extensions import TypedDict
from typing import Annotated
from langgraph.graph.message import add_messages

from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI

from langgraph.checkpoint.mongodb import MongoDBSaver
from langchain_core.messages import HumanMessage

import os

In [65]:
load_dotenv()

llm = ChatOpenAI(
    model="anthropic/claude-3-haiku",
)

In [66]:
class State(TypedDict):
    messages: Annotated[ list, add_messages ]

def chatbot_node(state : State):
    response = llm.invoke(state.get("messages"))
    return { "messages": [response] }

In [67]:
graph_builder = StateGraph(State)

graph_builder.add_node("chatbot_node", chatbot_node)

graph_builder.add_edge(START, "chatbot_node")

graph_builder.add_edge("chatbot_node", END)

graph = graph_builder.compile()

In [68]:
# Function to compile graph WITH MongoDB checkpointer
def compile_graph_with_checkpointer(checkpointer):

    # This enables conversation memory
    return graph_builder.compile(checkpointer=checkpointer)

In [69]:
DB_URI = os.environ.get("MONGODB_URI")

with MongoDBSaver.from_conn_string(DB_URI) as checkpointer:
    graph_with_checkpointer = compile_graph_with_checkpointer(checkpointer=checkpointer)

    config = {
        "configurable": {
            "thread_id" : "uuid"
        }
    }

    user_query = input("enter msg here: ")

    for chunk in graph_with_checkpointer.stream(
    #Initial stage we have to pass first msg here
    State({
        "messages": [HumanMessage (content= user_query)]
    }),

    config,
    stream_mode="values"
):
        chunk["messages"][-1].pretty_print()

================================ Human Message =================================

hi how are u
================================== Ai Message ==================================

I'm doing well, thanks for asking! As an AI assistant, I don't have personal experiences or feelings in the same way humans do. But I'm here and ready to help you with any questions or tasks you may have. Please let me know how I can be of assistance.
